# Resume Screening using NLP

In [2]:
# --- Install dependencies (only once)
!pip install streamlit sentence-transformers PyPDF2 pandas scikit-learn markovify -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.3 MB/s eta 0:00:00


# import Libraries

In [3]:
import os
import re
import pandas as pd
import numpy as np
import PyPDF2
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Setup Paths

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Path to your data folder
data_path = "/content/drive/MyDrive/data"
resume_path = "/content/drive/MyDrive/MLKholoudCV2Copy.pdf"

job_file = os.path.join(data_path, "data job posts.csv")


# Load Job Data


In [17]:
print(" Loading job dataset...")
jobs = pd.read_csv(job_file, encoding='utf-8', on_bad_lines='skip')

print(f" Loaded {len(jobs)} job postings.")
print("Available columns:", list(jobs.columns))

# Auto-detect job description column
job_col = next((c for c in jobs.columns if "description" in c.lower() or "responsibil" in c.lower()), None)
if not job_col:
    raise ValueError(" No job description column found.")
print(f" Using job description column: {job_col}")

 Loading job dataset...
 Loaded 19001 job postings.
Available columns: ['jobpost', 'date', 'Title', 'Company', 'AnnouncementCode', 'Term', 'Eligibility', 'Audience', 'StartDate', 'Duration', 'Location', 'JobDescription', 'JobRequirment', 'RequiredQual', 'Salary', 'ApplicationP', 'OpeningDate', 'Deadline', 'Notes', 'AboutC', 'Attach', 'Year', 'Month', 'IT']
 Using job description column: JobDescription


#Clean Text Function

In [18]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9.,!? ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Read Resume (PDF)

In [19]:
def read_pdf_text(pdf_path):
    """Extract text from PDF and clean it."""
    reader = PyPDF2.PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return clean_text(text)

print(" Reading resume...")
resume_text = read_pdf_text(resume_path)
print(" Resume loaded! Length:", len(resume_text), "characters")

 Reading resume...
 Resume loaded! Length: 5163 characters


#Load  Transforemer Model

In [10]:
print(" Loading SentenceTransformer model...")
model = SentenceTransformer('all-mpnet-base-v2')  # stronger than MiniLM
print(" Model loaded successfully!")

 Loading SentenceTransformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Model loaded successfully!


# Preprocess Job Descriptions

In [20]:
print(" Cleaning job descriptions...")
job_texts = jobs[job_col].astype(str).apply(clean_text).tolist()

 Cleaning job descriptions...


# Encode Resume

In [21]:
print(" Encoding job descriptions...")
job_embs = model.encode(job_texts, show_progress_bar=True, batch_size=32)
print(" Encoding resume...")
resume_emb = model.encode([resume_text])

 Encoding job descriptions...


Batches:   0%|          | 0/594 [00:00<?, ?it/s]

 Encoding resume...


# Compute Similarity

In [22]:
print(" Calculating similarity scores...")
sims = cosine_similarity(resume_emb, job_embs)[0]
jobs["match_score"] = sims

 Calculating similarity scores...


# Show Top Matches

In [23]:
top_jobs = jobs.sort_values("match_score", ascending=False).head(10)

print("\n Top Matching Jobs for Your Resume:\n")
for i, row in top_jobs.iterrows():
    level = (
        "Senior" if row["match_score"] > 0.75
        else "Junior" if row["match_score"] > 0.5
        else "Entry"
    )
    title = row.get("Title", row.get("Job Title", "Job"))
    print(f" {title}")
    print(f"   Match Score: {row['match_score']*100:.2f}%  |  Level: {level}")
    print(f"   Description: {str(row[job_col])[:250]}...")
    print("-" * 100)


 Top Matching Jobs for Your Resume:

 Lionbridge Internet Assessor
   Match Score: 45.73%  |  Level: Entry
   Description: The team at Lionbridge Technologies with solution
centres in 25 countries worldwide is recruiting part-time self-employed
workers who are fluent speakers in Armenian and English who are based in
Armenia to join its team of Internet Assessor.
The ...
----------------------------------------------------------------------------------------------------
 Internet Assessor
   Match Score: 44.73%  |  Level: Entry
   Description: The team at Lionbridge Technologies is currently
recruiting self-employed workers who are based in Armenia to join its
team of Internet Assessor. The main aim of the work is to improve a
search engines results for all web users worldwide. The work...
----------------------------------------------------------------------------------------------------
 Senior PHP Software Developer
   Match Score: 43.10%  |  Level: Entry
   Description: Key Ideas i

In [24]:
def extract_skills(text):
    """Extract common technical skills from resume."""
    skills = re.findall(
        r'\b(Python|Java|C\+\+|TensorFlow|SQL|AWS|Keras|Machine Learning|Deep Learning|Pandas|NumPy|Scikit-learn|PyTorch|Data Analysis|Cloud|API)\b',
        text, re.I
    )
    return sorted(set(s.capitalize() for s in skills))

print("\n Key Skills Found in Resume:")
print(", ".join(extract_skills(resume_text)) or "No skills found.")


 Key Skills Found in Resume:
Api, Aws, Data analysis, Deep learning, Java, Machine learning, Numpy, Pandas, Python, Pytorch, Sql, Tensorflow
